# SHREK Phase 2: GPU baseline handoff

This notebook detects hardware, verifies the software, and runs a **real local-model integration smoke test**. It does not claim research performance. The baseline controller is implemented; SHREK admission validation and maintenance belong to later phases.

Use **Runtime > Change runtime type > GPU** in Colab. The repository is private: download its ZIP while signed in to GitHub and upload that ZIP below. Do not put a GitHub token or model API key in notebook cells or outputs. On your PC, open this notebook from the project and skip the upload automatically.


In [ ]:
import importlib.util, json, platform, shutil, subprocess, sys
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], check=True)
else:
    print('No nvidia-smi command found. Confirm GPU runtime/hardware before loading a model.')


## Locate the source

In Colab, upload the private repository ZIP downloaded from GitHub (Code > Download ZIP). Archive paths are checked before extraction. This avoids saving authentication credentials in a Git remote. The ZIP name may be `SHREK-main.zip`.


In [ ]:
import zipfile
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').exists() and (p / 'src/shrek').exists()), None)
if ROOT is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one SHREK repository ZIP.')
    destination = Path('/content/shrek-source')
    destination.mkdir(exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.infolist():
            resolved = (destination / entry.filename).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise RuntimeError('Unsafe archive path')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError('Symbolic links are not accepted in source archives')
        archive.extractall(destination)
    roots = [p.parent for p in destination.rglob('pyproject.toml') if (p.parent / 'src/shrek').exists()]
    if len(roots) != 1:
        raise RuntimeError('Expected exactly one SHREK project in the archive.')
    ROOT = roots[0]
ROOT = ROOT.resolve()
print('Source:', ROOT)


## Install and verify

Installation downloads open-source dependencies; it does not call a paid inference service. Torch/CUDA compatibility depends on the machine. Keep the resulting package versions with your run. Installing on Colab uses its preinstalled Torch when compatible. The dependency ranges are not a hardware-tested lockfile.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT) + '[dev,hf]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'shrek', 'hardware'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'shrek', 'baseline-demo'], cwd=ROOT, check=True)


## Deliberate model choice

Review the hardware output first. `Qwen/Qwen2.5-1.5B-Instruct` is a **provisional integration candidate**, not a final research model or a guarantee of fit/quality. Its official card is https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct . Change it if your hardware and development checks justify another choice.

This configuration uses CUDA FP16, greedy decoding, context limit 4096, seed 17, and a 384-token output ceiling. Long or truncated structured outputs count as errors; there is no silent prompt truncation or automatic model substitution. Download/loading time can be significant. No fine-tuning occurs.


In [ ]:
from shrek.backends import TransformersBackend
from shrek.hardware import hardware_report
report = hardware_report()
print(json.dumps(report, indent=2))
if not report.get('cuda_available'):
    raise RuntimeError('Enable a working GPU runtime before this cell, or deliberately configure a CPU run outside this notebook.')
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # integration candidate only
MODEL_REVISION = 'main'  # use a model commit for frozen final experiments
backend = TransformersBackend(MODEL_ID, revision=MODEL_REVISION, device='cuda', context_limit=4096, seed=17)
print(json.dumps(backend.metadata(), indent=2))


## Real baseline smoke run

The two authored inputs are legitimate requests. This checks classification, host generation, feedback parsing, logging, and persistence without supplying benchmark labels to the model. It does **not** demonstrate attack resistance or natural rule learning. Inspect every error before expanding to published attack data.


In [ ]:
from datetime import datetime, timezone
from shrek.config import load_config
from shrek.data import load_examples, audit_splits
from shrek.runner import run_baseline
examples = load_examples(ROOT / 'examples/smoke.jsonl')
print(json.dumps(audit_splits(examples), indent=2))
OUTPUT = ROOT / 'runs' / ('real-smoke-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
summary = run_baseline(examples, backend, load_config(ROOT / 'configs/baseline_paper.toml'), OUTPUT,
                       learning=True, max_calls=20, max_output_tokens=384, token_counter=backend.count_tokens)
(OUTPUT / 'hardware.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(OUTPUT / 'environment-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
print(json.dumps(summary, indent=2))
if summary['events_with_errors']:
    print('CHECKPOINT NOT PASSED: inspect interactions.jsonl and calls.jsonl. Do not treat missing/invalid judgments as safe results.')
else:
    print('Integration smoke completed. Research model/judge calibration is still pending.')


## Inspect and export

Read `summary.json`, `manifest.json`, `interactions.jsonl`, and `manual_review.csv`. The original answers were saved before memory updates. The run ZIP can be returned for debugging; keep raw outputs local/ignored rather than committing generated completions. A repository ZIP has no Git metadata, so its manifest may have a null code commit: record the GitHub commit used to download it separately.


In [ ]:
print((OUTPUT / 'interactions.jsonl').read_text(encoding='utf-8'))
ARCHIVE = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
print('Run archive:', ARCHIVE)
try:
    from google.colab import files
    files.download(ARCHIVE)
except ImportError:
    print('Local notebook: retrieve the ZIP from the path above.')
